# Thiết lập môi trường QLoRA

Chọn **Runtime → Change runtime type → GPU**, ưu tiên T4. Dùng runtime có Python 3.12 hoặc 3.13. Nếu mặc định nằm ngoài khoảng này, chọn runtime có phiên bản Python phù hợp trong mục **Runtime Version**. [Danh sách runtime do Colab công bố](https://research.google.com/colaboratory/runtime-version-faq.html).

Chuẩn bị tệp `environment_colab_source.zip` đi kèm dự án. Các cell tải gói cài đặt và trọng số mô hình qua Internet. Môi trường ảo nằm trong `/content`, nên có thể chạy lại quy trình từ đầu khi Colab cấp máy ảo mới.

## 1. Xác nhận Python và GPU

Kiểm tra tên GPU, VRAM còn trống và driver trước khi tải PyTorch.

In [ ]:
from pathlib import Path
import platform
import shutil
import subprocess
import sys

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Chọn runtime Colab có Python 3.12 hoặc 3.13.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Hãy kết nối runtime GPU của Colab.")

print("Python", platform.python_version(), flush=True)
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"],
    check=True,
)

## 2. Nạp mã nguồn

Tải lên `environment_colab_source.zip` khi Colab hiện hộp chọn tệp. Có thể đặt sẵn tệp tại `/content/environment_colab_source.zip` trước khi chạy cell này.

In [ ]:
import io
from zipfile import ZipFile
from google.colab import files

PROJECT = Path("/content/traffic-law-qa")
archive_path = Path("/content/environment_colab_source.zip")
if archive_path.is_file():
    archive_data = archive_path.read_bytes()
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Chọn một tệp environment_colab_source.zip.")
    archive_data = next(iter(uploaded.values()))

required_files = (
    "requirements.txt",
    "configs/environment.toml",
    "scripts/check_environment.py",
)
with ZipFile(io.BytesIO(archive_data)) as archive:
    missing = set(required_files) - set(archive.namelist())
    if missing:
        raise ValueError(f"Tệp nguồn còn thiếu {sorted(missing)}.")
    for name in required_files:
        destination = PROJECT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(name))

## 3. Cài thư viện trong môi trường ảo

PyTorch dùng wheel CUDA 12.6. Cell sau chạy bằng Python trong môi trường ảo, vì vậy có thể giữ nguyên kernel Colab.

In [ ]:
VENV = PROJECT / ".venv"
PYTHON = VENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV)], check=True)
venv_pip = subprocess.run([str(PYTHON), "-m", "pip", "--version"], capture_output=True)
if venv_pip.returncode != 0:
    subprocess.run(
        [sys.executable, "-m", "pip", "--python", str(PYTHON), "install", "pip"], check=True
    )

subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-r", "requirements.txt"],
    cwd=PROJECT,
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)

## 4. Kiểm tra môi trường CUDA và NF4

Chạy thử forward và backward của một lớp bitsandbytes NF4 trên GPU. Tệp JSON ghi phiên bản thư viện, cấu hình GPU và precision thực tế.

In [ ]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = PROJECT / "artifacts" / "environment_checks" / run_id
subprocess.run(
    [str(PYTHON), "scripts/check_environment.py", "environment",
     "--output", str(RUN_DIR / "environment.json")],
    cwd=PROJECT,
    check=True,
)

## 5. Kiểm tra QLoRA trên Qwen

Để `primary` khi xác nhận Qwen2.5-3B-Instruct trên Colab. Nếu GPU thiếu VRAM, có thể chọn `fallback` để kiểm tra Qwen2.5-1.5B-Instruct, rồi chạy lại `primary` khi có GPU phù hợp.

Cell sinh một đoạn văn bản ngắn và chạy một bước SFT trên câu chào tiếng Việt. Kiểm tra chỉ LoRA được cập nhật và adapter lưu tạm có thể nạp lại. Loss trong JSON thuộc phép thử môi trường này.

In [ ]:
MODEL = "primary"  # @param ["primary", "fallback"]
attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
subprocess.run(
    [str(PYTHON), "scripts/check_environment.py", "qlora", "--model", MODEL,
     "--output", str(RUN_DIR / f"qlora-{MODEL}-{attempt}.json")],
    cwd=PROJECT,
    check=True,
)

## 6. Tải bằng chứng kiểm thử

Lưu ZIP kết quả từ phiên Colab để đối chiếu GPU, revision của Qwen và các bước đã vượt qua.

In [ ]:
report_archive = shutil.make_archive(
    str(Path("/content") / f"environment-check-{run_id}"), "zip", root_dir=RUN_DIR
)
files.download(report_archive)